# Validando argumentos de funções

Um uso possível do Pydantic, além de validar dados serializados/desserializados, é validar os **argumentos passados a uma função**.

Isso não é checagem estática de tipos — qualquer exceção de validação acontece em tempo de execução; isso é coisa do Python, não do Pydantic.

Considere este exemplo bem simples: uma função que exige um argumento string, que não pode ser vazio (nem `None`).

Em Python puro, poderíamos escrever assim:

In [1]:
def extract_first_char(s: str):
    if s is None:
        raise ValueError("argument cannot be None")
    if not isinstance(s, str):
        raise TypeError("argument must be a string")
    if len(s) == 0:
        raise ValueError("argument cannot be an empty string")
    return s[0]

In [2]:
try:
    extract_first_char(100)
except TypeError as ex:
    print(f"{type(ex).__name__}: {ex}")

TypeError: argument must be a string


In [3]:
try:
    extract_first_char("")
except ValueError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValueError: argument cannot be an empty string


Também dá para simplificar um pouco essas validações:

In [4]:
def extract_first_char(s: str):
    if not isinstance(s, str):
        raise TypeError("argument must be a string")
    if not s:
        raise ValueError("argument cannot be an empty string")
    return s[0]

In [5]:
try:
    extract_first_char(100)
except TypeError as ex:
    print(f"{type(ex).__name__}: {ex}")

TypeError: argument must be a string


In [6]:
try:
    extract_first_char("")
except ValueError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValueError: argument cannot be an empty string


In [7]:
try:
    extract_first_char(None)
except TypeError as ex:
    print(f"{type(ex).__name__}: {ex}")

TypeError: argument must be a string


Pensando bem, toda essa validação é o que o Pydantic já faz quando definimos modelos.

E o Pydantic nos deixa reaproveitar essa lógica de validação nos argumentos de funções.

Primeiro, um tipo `Annotated` que define o que o argumento `s` precisa ser, além de string.

In [8]:
from typing import Annotated

from pydantic import Field, ValidationError

In [9]:
NonEmptyString= Annotated[str, Field(min_length=1)]

Isto é só um tipo `Annotated` comum, e podemos usá-lo na função:

In [10]:
def extract_first_char(s: NonEmptyString):
    if s is None:
        raise ValueError("argument cannot be None")
    if not isinstance(s, str):
        raise TypeError("argument must be a string")
    if len(s) == 0:
        raise ValueError("argument cannot be an empty string")
    return s[0]

Claro, isso não muda nada do nosso código de validação (anotações não são verificadas em tempo de execução).

Mas podemos usar um decorador especial do Pydantic que valida os argumentos quando a função é chamada:

In [11]:
from pydantic import validate_call

In [12]:
@validate_call
def extract_first_char(s: NonEmptyString):
    return s[0]

Removemos todo o código de validação: ao chamar `extract_first_char()`, chamamos na verdade a função **decorada**, que primeiro roda os validadores do Pydantic e só depois chama a nossa `extract_first_char` original:

In [13]:
print(f'extract_first_char("abc"): {extract_first_char("abc")!r}')

extract_first_char("abc"): 'a'


In [14]:
try:
    extract_first_char(None)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for extract_first_char
0
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


In [15]:
try:
    extract_first_char("")
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for extract_first_char
0
  String should have at least 1 character [type=string_too_short, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short


In [16]:
try:
    extract_first_char(100)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for extract_first_char
0
  Input should be a valid string [type=string_type, input_value=100, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type


Note que a exceção não é capturada **dentro** da função, e sim no código que a chama. Então isso não serve para mudar o comportamento dentro da função — o uso mais comum é permitir uma ação corretiva antes de chamar a função, ou apresentar os erros de validação ao usuário da aplicação. Isso é bem útil em APIs REST, por exemplo, em que os erros de validação podem ser devolvidos como algum erro HTTP.

Vejamos como usar também um validador customizado.

In [17]:
from datetime import datetime
from typing import Any

import pytz
from dateutil.parser import parse

def make_utc(dt: datetime) -> datetime:
    print("make_utc called...")
    if dt.tzinfo is None:
        dt = pytz.utc.localize(dt)
    else:
        dt = dt.astimezone(pytz.utc)
    return dt
    
def parse_datetime(value: Any):
    print("parse_datetime called...")
    if isinstance(value, str):
        try:
            return parse(value)
        except Exception as ex:
            raise ValueError(str(ex))
    return value

In [18]:
from pydantic import BeforeValidator, AfterValidator

DatetimeUTC = Annotated[datetime, BeforeValidator(parse_datetime), AfterValidator(make_utc)]

In [19]:
@validate_call
def func(dt: DatetimeUTC):
    return dt.isoformat()

In [20]:
print(f'func("2020/1/1 3pm"): {func("2020/1/1 3pm")!r}')

parse_datetime called...
make_utc called...
func("2020/1/1 3pm"): '2020-01-01T15:00:00+00:00'


A validação funciona exatamente como nos campos de modelos: os validadores não só validam, como também **modificam** o dado conforme ele passa pelo pipeline before/after.

Então essa é uma forma prática não só de validar argumentos, mas também de transformá-los a partir da entrada bruta.

Um aviso, porém: se você ainda não usa Pydantic na aplicação, pense duas vezes antes de adicioná-lo como dependência só para validar argumentos de funções — pode ser a escolha certa, mas pode não ser; numa aplicação pontual, talvez Python puro seja mais eficaz:

In [21]:
def func(dt: datetime | str):
    try:
        dt = parse(dt)
    except Exception as ex:
        raise ValueError(str(ex))
    dt = make_utc(dt)
    return dt

In [22]:
print(f'func("2020/1/1 3pm"): {func("2020/1/1 3pm")!r}')

make_utc called...
func("2020/1/1 3pm"): datetime.datetime(2020, 1, 1, 15, 0, tzinfo=<UTC>)


> 📝 **Nota de revisão — bug na versão "Python puro":** a anotação diz `datetime | str`, mas `parse()` só aceita string. Passar um `datetime` (metade do contrato declarado) quebra:

In [23]:
try:
    func(datetime(2020, 1, 1, 15))
except Exception as ex:
    print(f"{type(ex).__name__}: {ex}")

ValueError: Parser must be a string or character stream, not datetime


Corrigindo — só faz parse quando é string:

In [24]:
def func(dt: datetime | str) -> datetime:
    if isinstance(dt, str):
        try:
            dt = parse(dt)
        except Exception as ex:
            raise ValueError(str(ex)) from ex
    elif not isinstance(dt, datetime):
        raise TypeError(f"expected datetime or str, got {type(dt).__name__}")
    return make_utc(dt)

print(f"func('2020/1/1 3pm'): {func('2020/1/1 3pm')!r}")
print(f"func(datetime(2020, 1, 1, 15)): {func(datetime(2020, 1, 1, 15))!r}")

make_utc called...
func('2020/1/1 3pm'): datetime.datetime(2020, 1, 1, 15, 0, tzinfo=<UTC>)
make_utc called...
func(datetime(2020, 1, 1, 15)): datetime.datetime(2020, 1, 1, 15, 0, tzinfo=<UTC>)


Repare que a versão em Python puro, para ficar correta, voltou a ter o mesmo tipo de código defensivo do início da aula — é exatamente esse código que o `@validate_call` nos poupa.

> 💡 **Complemento — `validate_return` e argumentos múltiplos:** o `@validate_call` também pode validar o **retorno** (`validate_return=True`), e o erro aponta qual argumento falhou (pela posição ou pelo nome; um erro no **retorno** vem com `loc=()`):

In [25]:
@validate_call(validate_return=True)
def initials(first: NonEmptyString, last: NonEmptyString, *, sep: str = ".") -> NonEmptyString:
    return f"{first[0]}{sep}{last[0]}{sep}" if sep else ""

print(f"initials('Ada', 'Lovelace'): {initials('Ada', 'Lovelace')!r}")
for args, kwargs in [(("Ada", ""), {}), (("Ada", "Lovelace"), {"sep": 1}), (("Ada", "Lovelace"), {"sep": ""})]:
    try:
        initials(*args, **kwargs)
    except ValidationError as ex:
        error = ex.errors()[0]
        print(f"initials{args} {kwargs}: loc={error['loc']} -> {error['msg']}")

initials('Ada', 'Lovelace'): 'A.L.'
initials('Ada', '') {}: loc=(1,) -> String should have at least 1 character
initials('Ada', 'Lovelace') {'sep': 1}: loc=('sep',) -> Input should be a valid string
initials('Ada', 'Lovelace') {'sep': ''}: loc=() -> String should have at least 1 character
